In [1]:
import polars as pl
from polars import col
from investment_strategy.data.cleaner import *
from investment_strategy.signals.signal_construction import *
from investment_strategy.signals.signal_ranking import *
from investment_strategy.portfolio.weighting import *
from investment_strategy.portfolio.rebalancing import *
from investment_strategy.portfolio.valuation import *
from investment_strategy.analytics.return_metrics import *
from investment_strategy.analytics.risk_metrics import *
from investment_strategy.config.backtest_config import *
from investment_strategy.analytics.turnover_metrics import *
from investment_strategy.analytics.benchmark_metrics import *
from investment_strategy.portfolio.construction import *
from datetime import date

market_data = pl.read_parquet("../data/raw/sp500_market_data.parquet")
market_data

date,ticker,open,high,low,close,volume
date,str,f64,f64,f64,f64,i64
2018-12-31,"""A""",62.795802,63.874904,62.795802,63.855968,1572100
2018-12-31,"""AAPL""",37.613949,37.810881,37.127551,37.42651,140014000
2018-12-31,"""ABBV""",66.040205,67.042343,65.773452,66.465576,5722100
2018-12-31,"""ABNB""",null,null,null,null,null
2018-12-31,"""ABT""",62.168729,63.202416,62.090555,62.828899,6094300
…,…,…,…,…,…,…
2026-05-29,"""XYZ""",74.970001,76.660004,74.195,75.720001,7380400
2026-05-29,"""YUM""",149.229996,150.179993,147.339996,147.949997,3992700
2026-05-29,"""ZBH""",81.952229,82.979498,81.363796,82.111809,3216600


In [2]:
benchmark_data = pl.read_parquet("../data/raw/benchmark_market_data.parquet")
benchmark_data

date,ticker,open,high,low,close,volume
date,str,f64,f64,f64,f64,i64
2018-12-31,"""^DJI""",23153.939453,23333.179688,23118.300781,23327.460938,288830000
2018-12-31,"""^GSPC""",2498.939941,2509.23999,2482.820068,2506.850098,3461920000
2018-12-31,"""^IXIC""",6649.52002,6659.959961,6570.060059,6635.279785,2109320000
2018-12-31,"""^NDX""",6354.850098,6365.390137,6273.939941,6329.970215,2109320000
2018-12-31,"""^RUT""",1338.52002,1348.560059,1329.170044,1348.560059,3461920000
…,…,…,…,…,…,…
2026-05-29,"""^DJI""",50773.910156,51094.179688,50698.269531,51032.460938,894700000
2026-05-29,"""^GSPC""",7579.330078,7599.379883,7563.549805,7580.060059,7858290000
2026-05-29,"""^IXIC""",26960.839844,27094.800781,26859.269531,26972.619141,11906000000


# Signal Construction

In [3]:
market_data = fill_OHLCV_missing_values(market_data)
close_price = market_data.select(
    col("date"),
    col("ticker"),
    col("close")
)

In [4]:
end_date = get_backtest_end_date(
    backtest_start_date=BACKTEST_START_DATE,
    backtest_period=BACKTEST_PERIOD,
    backtest_period_unit=BACKTEST_PERIOD_UNIT,
)
end_date

datetime.date(2024, 12, 31)

In [5]:
trading_calendar = get_trading_calendar(cleaned_close_prices_dataset=close_price)
trading_calendar

date
date
2018-12-31
2019-01-02
2019-01-03
2019-01-04
2019-01-07
…
2026-05-22
2026-05-26
2026-05-27


In [6]:
date_mapping_df = create_date_mapping(
    trading_calendar=trading_calendar,
    rebalance_frequency=REBALANCE_FREQUENCY,
    rebalance_freq_unit=REBALANCE_FREQ_UNIT,
    backtest_start_date=BACKTEST_START_DATE,
    backtest_end_date=end_date,
    lookback_period_for_total_returns=LOOKBACK_PERIOD,
    lookback_period_unit=LOOKBACK_PERIOD_UNIT,
)
date_mapping_df

lookback_date,lag_base_date,signal_date,rebalance_date
date,date,date,date
2021-06-30,2021-11-30,2021-12-30,2021-12-31
2021-08-25,2022-01-25,2022-02-25,2022-02-28
2021-10-29,2022-03-29,2022-04-29,2022-05-02
2021-12-29,2022-05-27,2022-06-29,2022-06-30
2022-02-28,2022-07-29,2022-08-30,2022-08-31
…,…,…,…
2023-10-27,2024-03-28,2024-04-29,2024-04-30
2023-12-28,2024-05-28,2024-06-28,2024-07-01
2024-02-29,2024-07-30,2024-08-30,2024-09-03


In [7]:
prices_for_date_mapping = get_prices_for_date_mapping(
    cleaned_close_prices_dataset=close_price,
    cleaned_stock_OHLCV=market_data,
    date_mapping_df=date_mapping_df,
)
prices_for_date_mapping

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open
date,str,f64,date,date,date,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822
…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847


In [8]:
momentum = calculate_momentum(
    factor_reference_table=prices_for_date_mapping,
    lookback_period_for_total_returns=LOOKBACK_PERIOD,
    lookback_period_unit=LOOKBACK_PERIOD_UNIT,
)
momentum

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo
date,str,f64,date,date,date,f64,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484,0.02352
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375,0.210494
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431,0.047393
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999,0.126681
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822,0.093214
…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001,0.373081
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221,0.054154
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847,0.03519


In [9]:
past_std = calculate_past_returns_std(
    cleaned_close_prices_dataset=close_price,
    factor_reference_table=momentum,
    date_mapping_df=date_mapping_df,
    lookback_period_for_total_returns=LOOKBACK_PERIOD,
    lookback_period_unit=LOOKBACK_PERIOD_UNIT,
)
past_std

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo
date,str,f64,date,date,date,f64,f64,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484,0.02352,0.01249
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375,0.210494,0.013089
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431,0.047393,0.012661
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999,0.126681,0.027615
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822,0.093214,0.010467
…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001,0.373081,0.027146
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221,0.054154,0.010647
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847,0.03519,0.015842


In [10]:
risk_adjusted_table = get_risk_adjusted_return(
    factor_reference_with_momentum_and_vol=past_std,
    lookback_period_for_total_returns=LOOKBACK_PERIOD,
    lookback_period_unit=LOOKBACK_PERIOD_UNIT,
)
risk_adjusted_table

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484,0.02352,0.01249,1.883082
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375,0.210494,0.013089,16.081247
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431,0.047393,0.012661,3.743267
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999,0.126681,0.027615,4.587432
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822,0.093214,0.010467,8.905733
…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001,0.373081,0.027146,13.743568
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221,0.054154,0.010647,5.086498
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847,0.03519,0.015842,2.221282


# Signal Ranking

In [11]:
signal_ranked = rank_signal(
    signal_df=risk_adjusted_table,
    signal_col=f"risk_adjusted_momentum_{LOOKBACK_PERIOD}{LOOKBACK_PERIOD_UNIT}",
)
signal_ranked

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo,risk_adjusted_momentum_6mo rank
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64,u32
2021-12-30,"""A""",155.463638,2021-06-30,2021-11-30,2021-12-31,142.468719,145.819595,154.951484,0.02352,0.01249,1.883082,255
2021-12-30,"""AAPL""",174.214935,2021-06-30,2021-11-30,2021-12-31,133.502045,161.603409,174.107375,0.210494,0.013089,16.081247,42
2021-12-30,"""ABBV""",114.511681,2021-06-30,2021-11-30,2021-12-31,92.721123,97.115463,114.60431,0.047393,0.012661,3.743267,213
2021-12-30,"""ABNB""",168.779999,2021-06-30,2021-11-30,2021-12-31,153.139999,172.539993,168.779999,0.126681,0.027615,4.587432,195
2021-12-30,"""ABT""",128.427795,2021-06-30,2021-11-30,2021-12-31,104.788086,114.555786,128.427822,0.093214,0.010467,8.905733,111
…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""XYZ""",87.480003,2024-06-28,2024-11-29,2024-12-31,64.489998,88.550003,87.720001,0.373081,0.027146,13.743568,191
2024-12-30,"""YUM""",129.748352,2024-06-28,2024-11-29,2024-12-31,127.461578,134.364136,130.30221,0.054154,0.010647,5.086498,341
2024-12-30,"""ZBH""",103.810539,2024-06-28,2024-11-29,2024-12-31,106.416763,110.161545,104.312847,0.03519,0.015842,2.221282,376


In [12]:
rank_col = f"risk_adjusted_momentum_{LOOKBACK_PERIOD}{LOOKBACK_PERIOD_UNIT} rank"

In [13]:
ranked_candidates = filter_ranked_candidates(
    ranked_signal_df=signal_ranked,
    rank_col=rank_col,
    rank_buffer=RANK_BUFFER,
)
ranked_candidates

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo,risk_adjusted_momentum_6mo rank
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64,u32
2021-12-30,"""ALB""",221.557343,2021-06-30,2021-11-30,2021-12-31,158.740082,251.533096,221.235862,0.584559,0.02389,24.468939,10
2021-12-30,"""AMD""",145.149994,2021-06-30,2021-11-30,2021-12-31,93.93,158.369995,146.160004,0.686043,0.025971,26.415665,8
2021-12-30,"""BLDR""",84.050003,2021-06-30,2021-11-30,2021-12-31,42.66,69.440002,84.290001,0.627754,0.020418,30.7454,3
2021-12-30,"""BX""",109.858231,2021-06-30,2021-11-30,2021-12-31,81.903946,120.929535,109.832578,0.47648,0.020521,23.219677,14
2021-12-30,"""COST""",535.266479,2021-06-30,2021-11-30,2021-12-31,374.264069,511.982544,534.791945,0.367971,0.010224,35.990245,2
…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""TRGP""",172.180573,2024-06-28,2024-11-29,2024-12-31,123.482193,197.887573,172.538972,0.60256,0.016254,37.070911,6
2024-12-30,"""UAL""",98.209999,2024-06-28,2024-11-29,2024-12-31,48.66,96.830002,97.82,0.98993,0.028557,34.665381,10
2024-12-30,"""WEC""",89.579071,2024-06-28,2024-11-29,2024-12-31,73.274673,96.08284,89.579061,0.311269,0.009638,32.295004,15


In [14]:
sorted_ranking = sort_rankings(
    ranked_signal_df=ranked_candidates,
    rank_col=f"risk_adjusted_momentum_{LOOKBACK_PERIOD}{LOOKBACK_PERIOD_UNIT} rank",
)
sorted_ranking

signal_date,ticker,signal_close,lookback_date,lag_base_date,rebalance_date,lookback_close,lag_base_close,rebalance_open,momentum_6mo,vol_6mo,risk_adjusted_momentum_6mo,risk_adjusted_momentum_6mo rank
date,str,f64,date,date,date,f64,f64,f64,f64,f64,f64,u32
2021-12-30,"""FDS""",461.847412,2021-06-30,2021-11-30,2021-12-31,318.493347,446.441071,461.847424,0.401728,0.010503,38.250071,1
2021-12-30,"""COST""",535.266479,2021-06-30,2021-11-30,2021-12-31,374.264069,511.982544,534.791945,0.367971,0.010224,35.990245,2
2021-12-30,"""BLDR""",84.050003,2021-06-30,2021-11-30,2021-12-31,42.66,69.440002,84.290001,0.627754,0.020418,30.7454,3
2021-12-30,"""VRSK""",221.089966,2021-06-30,2021-11-30,2021-12-31,168.90416,217.692719,220.925249,0.288854,0.009398,30.735613,4
2021-12-30,"""ODFL""",173.97641,2021-06-30,2021-11-30,2021-12-31,123.783798,173.439255,173.854314,0.401147,0.013245,30.28613,5
…,…,…,…,…,…,…,…,…,…,…,…,…
2024-12-30,"""FOX""",45.368912,2024-06-28,2024-11-29,2024-12-31,31.302053,44.039742,45.36891,0.406928,0.012618,32.249867,16
2024-12-30,"""WMT""",89.35701,2024-06-28,2024-11-29,2024-12-31,66.466034,91.05941,89.357008,0.370014,0.011513,32.138424,17
2024-12-30,"""GEV""",329.117523,2024-06-28,2024-11-29,2024-12-31,170.788254,332.71402,329.785165,0.948108,0.029675,31.949356,18


# Portfolio Construction

In [15]:
rebalance_date = date_mapping_df.get_column("rebalance_date")
rebalance_date

rebalance_date
date
2021-12-31
2022-02-28
2022-05-02
2022-06-30
2022-08-31
…
2024-04-30
2024-07-01
2024-09-03


In [16]:
portfolio_basket = construct_portfolio(
    ranked_candidates=ranked_candidates,
    rebalance_dates=rebalance_date,
    rank_col=rank_col,
    top_n=TOP_N,
)
portfolio_basket

rebalance_date,ticker,rebalance_open,risk_adjusted_momentum_6mo rank
date,str,f64,u32
2021-12-31,"""ALB""",221.235862,10
2021-12-31,"""AMD""",146.160004,8
2021-12-31,"""BLDR""",84.290001,3
2021-12-31,"""COST""",534.791945,2
2021-12-31,"""DDOG""",179.190002,6
…,…,…,…
2024-12-31,"""FISV""",206.770004,4
2024-12-31,"""NI""",35.274594,5
2024-12-31,"""TRGP""",172.538972,6


# Portfolio Weights

In [17]:
equal_weighted_portfolio = construct_portfolio_weights(
    portfolio_basket=portfolio_basket, weighting_method="equal_weighted"
)
equal_weighted_portfolio

rebalance_date,ticker,rebalance_open,risk_adjusted_momentum_6mo rank,portfolio_weight
date,str,f64,u32,f64
2021-12-31,"""ALB""",221.235862,10,0.1
2021-12-31,"""AMD""",146.160004,8,0.1
2021-12-31,"""BLDR""",84.290001,3,0.1
2021-12-31,"""COST""",534.791945,2,0.1
2021-12-31,"""DDOG""",179.190002,6,0.1
…,…,…,…,…
2024-12-31,"""FISV""",206.770004,4,0.1
2024-12-31,"""NI""",35.274594,5,0.1
2024-12-31,"""TRGP""",172.538972,6,0.1


In [18]:
rebalance_allocation_df = prepare_rebalance_allocation_df(
    weighted_portfolio_signal_df=equal_weighted_portfolio
)
rebalance_allocation_df

rebalance_date,ticker,rebalance_open,portfolio_weight
date,str,f64,f64
2021-12-31,"""ALB""",221.235862,0.1
2021-12-31,"""AMD""",146.160004,0.1
2021-12-31,"""BLDR""",84.290001,0.1
2021-12-31,"""COST""",534.791945,0.1
2021-12-31,"""DDOG""",179.190002,0.1
…,…,…,…
2024-12-31,"""FISV""",206.770004,0.1
2024-12-31,"""NI""",35.274594,0.1
2024-12-31,"""TRGP""",172.538972,0.1


# Rebalance Simulation

In [19]:
rebalance_summary = run_rebalance_simulation(
    factor_reference_table=prices_for_date_mapping,
    rebalance_allocation_df=rebalance_allocation_df,
    initial_capital=INITIAL_CAPITAL,
    rebalance_dates=rebalance_date,
    execution_cost_rate=EXECUTION_COST_RATE,
    commission_per_share=COMMISSION_PER_SHARE
)
rebalance_summary["rebalance_level_table"]

rebalance_date,pre_rebalance_portfolio_value,post_rebalance_portfolio_value,cash_residual,transaction_cost
date,f64,f64,f64,f64
2021-12-31,1e6,998976.191123,1137.314027,1023.808877
2022-02-28,856221.430925,854600.649137,-243.427515,1620.781788
2022-05-02,928236.931893,926643.088029,-75.345473,1593.843863
2022-06-30,881639.637994,880471.309757,313.848055,1168.328237
2022-08-31,989024.769776,987755.288286,556.463107,1269.481489
…,…,…,…,…
2024-04-30,1.7936e6,1.7909e6,-32.301471,2650.055038
2024-07-01,1.8629e6,1.8601e6,-96.048187,2752.567892
2024-09-03,1.8786e6,1.8745e6,-1324.519057,4018.695079


In [20]:
rebalance_summary["position_level_table"]

rebalance_date,ticker,shares
date,str,i64
2021-12-31,"""ALB""",451
2021-12-31,"""AMD""",683
2021-12-31,"""BLDR""",1185
2021-12-31,"""COST""",186
2021-12-31,"""DDOG""",557
…,…,…
2024-12-31,"""FISV""",930
2024-12-31,"""NI""",5454
2024-12-31,"""TRGP""",1115


In [21]:
rebalance_summary["trade_level_table"]

rebalance_date,ticker,shares_traded,rebalance_open,trade_value,execution_cost,commission,transaction_cost,cash_flows
date,str,i64,f64,f64,f64,f64,f64,f64
2021-12-31,"""ALB""",451,221.235862,99777.373681,99.777374,2.255,102.032374,-99879.406055
2021-12-31,"""AMD""",683,146.160004,99827.282501,99.827283,3.415,103.242283,-99930.524784
2021-12-31,"""BLDR""",1185,84.290001,99883.651085,99.883651,5.925,105.808651,-99989.459736
2021-12-31,"""COST""",186,534.791945,99471.301821,99.471302,0.93,100.401302,-99571.703122
2021-12-31,"""DDOG""",557,179.190002,99808.83136,99.808831,2.785,102.593831,-99911.425191
…,…,…,…,…,…,…,…,…
2024-12-31,"""FISV""",930,206.770004,192296.103973,192.296104,4.65,196.946104,-192493.050077
2024-12-31,"""NI""",5454,35.274594,192387.636841,192.387637,27.27,219.657637,-192607.294478
2024-12-31,"""TRGP""",1115,172.538972,192380.953984,192.380954,5.575,197.955954,-192578.909938


# Portfolio Valuation

In [22]:
rebalance_period_close_prices = get_backtest_period_close_prices(
    cleaned_close_prices_dataset=close_price,
    backtest_start_date=BACKTEST_START_DATE,
    backtest_end_date=end_date,
)
rebalance_period_close_prices

date,ticker,close
date,str,f64
2021-12-31,"""A""",154.27504
2021-12-31,"""AAPL""",173.599014
2021-12-31,"""ABBV""",114.065155
2021-12-31,"""ABNB""",166.490005
2021-12-31,"""ABT""",128.19101
…,…,…
2024-12-31,"""XYZ""",84.989998
2024-12-31,"""YUM""",130.370239
2024-12-31,"""ZBH""",104.037064


In [23]:
next_date_matched_rebalance_level_table = get_next_date_matched_rebalance_level_table(
    rebalance_level_table=rebalance_summary["rebalance_level_table"]
)
next_date_matched_rebalance_level_table

rebalance_date,pre_rebalance_portfolio_value,post_rebalance_portfolio_value,cash_residual,transaction_cost,next_rebalance_date
date,f64,f64,f64,f64,date
2021-12-31,1e6,998976.191123,1137.314027,1023.808877,2022-02-28
2022-02-28,856221.430925,854600.649137,-243.427515,1620.781788,2022-05-02
2022-05-02,928236.931893,926643.088029,-75.345473,1593.843863,2022-06-30
2022-06-30,881639.637994,880471.309757,313.848055,1168.328237,2022-08-31
2022-08-31,989024.769776,987755.288286,556.463107,1269.481489,2022-10-31
…,…,…,…,…,…
2024-04-30,1.7936e6,1.7909e6,-32.301471,2650.055038,2024-07-01
2024-07-01,1.8629e6,1.8601e6,-96.048187,2752.567892,2024-09-03
2024-09-03,1.8786e6,1.8745e6,-1324.519057,4018.695079,2024-10-31


In [24]:
daily_position_value_table = get_daily_position_value_table(
    backtest_period_close_prices=rebalance_period_close_prices,
    next_date_matched_rebalance_level_table=next_date_matched_rebalance_level_table,
    position_level_table=rebalance_summary["position_level_table"],
)
daily_position_value_table

date,rebalance_date,ticker,shares,close,position_value
date,date,str,i64,f64,f64
2021-12-31,2021-12-31,"""ALB""",451,221.008972,99675.046448
2021-12-31,2021-12-31,"""AMD""",683,143.899994,98283.695831
2021-12-31,2021-12-31,"""BLDR""",1185,85.709999,101566.348915
2021-12-31,2021-12-31,"""COST""",186,538.864075,100228.717896
2021-12-31,2021-12-31,"""DDOG""",557,178.110001,99207.27034
…,…,…,…,…,…
2024-12-31,2024-12-31,"""FOX""",4241,45.034157,190989.858986
2024-12-31,2024-12-31,"""NI""",5454,35.284191,192439.978432
2024-12-31,2024-12-31,"""PLTR""",2480,75.629997,187562.393188


In [25]:
daily_portfolio_table = get_daily_portfolio_table(
    next_date_matched_rebalance_level_table=next_date_matched_rebalance_level_table,
    daily_position_value_table=daily_position_value_table,
    initial_capital=INITIAL_CAPITAL
)
daily_portfolio_table

date,positions_value,cash_residual,portfolio_value,daily_return
date,f64,f64,f64,f64
2021-12-31,999775.987938,1137.314027,1.0009e6,0.000913
2022-01-03,984317.072136,1137.314027,985454.386163,-0.015445
2022-01-04,979994.58197,1137.314027,981131.895998,-0.004386
2022-01-05,939071.453529,1137.314027,940208.767557,-0.04171
2022-01-06,936567.684242,1137.314027,937704.99827,-0.002663
…,…,…,…,…
2024-12-24,1.9577e6,1714.279146,1.9594e6,0.006932
2024-12-26,1.9525e6,1714.279146,1.9542e6,-0.002668
2024-12-27,1.9381e6,1714.279146,1.9398e6,-0.007385


# Analytics

## Return metrics

In [26]:
daily_portfolio_value_return_df = prepare_daily_portfolio_value_return_df(
    daily_portfolio_table=daily_portfolio_table
)
daily_portfolio_value_return_df

date,portfolio_value,daily_return
date,f64,f64
2021-12-31,1.0009e6,0.000913
2022-01-03,985454.386163,-0.015445
2022-01-04,981131.895998,-0.004386
2022-01-05,940208.767557,-0.04171
2022-01-06,937704.99827,-0.002663
…,…,…
2024-12-24,1.9594e6,0.006932
2024-12-26,1.9542e6,-0.002668
2024-12-27,1.9398e6,-0.007385


In [27]:
total_return = calculate_total_return(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df,
    initial_capital=INITIAL_CAPITAL
)
total_return

0.9039075737423574

In [28]:
annualized_return_cagr = calculate_annualized_return_cagr(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df,
    initial_capital=INITIAL_CAPITAL
)
annualized_return_cagr

0.24047113457023728

In [29]:
mean_daily_return = calculate_mean_daily_return(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
mean_daily_return

0.0009766210963434694

In [30]:
annualized_mean_daily_return = calculate_annualized_mean_return(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
annualized_mean_daily_return

0.2788847602386313

## Risk metrics

In [31]:
mean_daily_std = calculate_daily_return_std(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
mean_daily_std

0.015632371630477166

In [32]:
annualized_volatility = calculate_annualized_volatility(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
annualized_volatility

0.24815620641830324

In [33]:
drawdown = calculate_drawdown(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
drawdown

date,portfolio_value,daily_return,drawdown
date,f64,f64,f64
2021-12-31,1.0009e6,0.000913,0.0
2022-01-03,985454.386163,-0.015445,-0.015445
2022-01-04,981131.895998,-0.004386,-0.019763
2022-01-05,940208.767557,-0.04171,-0.060649
2022-01-06,937704.99827,-0.002663,-0.063151
…,…,…,…
2024-12-24,1.9594e6,0.006932,-0.052026
2024-12-26,1.9542e6,-0.002668,-0.054556
2024-12-27,1.9398e6,-0.007385,-0.061538


In [34]:
max_drawdown = calculate_max_drawdown(drawdown_table=drawdown)
max_drawdown

-0.20134209990266316

In [35]:
sharpe_ratio = calculate_sharpe_ratio(
    mean_daily_return=mean_daily_return, annualized_volatility=annualized_volatility, annual_rf=ANNUAL_RISK_FREE_RATE
)
sharpe_ratio

0.8708567857225866

In [36]:
calmer_ratio = calculate_calmer_ratio(
    annualized_return_cagr=annualized_return_cagr, max_drawdown=max_drawdown
)
calmer_ratio

1.1943410478309835

In [37]:
sortino_ratio = calculate_sortino_ratio(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df,
    mean_daily_return=mean_daily_return,
    annual_rf=ANNUAL_RISK_FREE_RATE
)
sortino_ratio

1.2505260424192597

## Turnover metrics

In [38]:
portfolio_turnover_table = get_portfolio_turnover_table(
    rebalance_level_table=rebalance_summary["rebalance_level_table"],
    trade_level_table=rebalance_summary["trade_level_table"],
)
portfolio_turnover_table

rebalance_date,buy_value,sell_value,pre_rebalance_portfolio_value,post_rebalance_portfolio_value,cash_residual,transaction_cost,one_way_turnover,transaction_cost_ratio
date,f64,f64,f64,f64,f64,f64,f64,f64
2021-12-31,997838.877096,0.0,1e6,998976.191123,1137.314027,1023.808877,1.0,0.001024
2022-02-28,769380.874055,769620.9143,856221.430925,854600.649137,-243.427515,1620.781788,0.898577,0.001893
2022-05-02,741280.968753,743042.89466,928236.931893,926643.088029,-75.345473,1593.843863,0.79859,0.001717
2022-06-30,534540.3576,536097.879365,881639.637994,880471.309757,313.848055,1168.328237,0.606303,0.001325
2022-08-31,593047.196311,594559.292852,989024.769776,987755.288286,556.463107,1269.481489,0.599628,0.001284
…,…,…,…,…,…,…,…,…
2024-04-30,1.2708e6,1.2733e6,1.7936e6,1.7909e6,-32.301471,2650.055038,0.708519,0.001478
2024-07-01,1.3123e6,1.3150e6,1.8629e6,1.8601e6,-96.048187,2752.567892,0.704459,0.001478
2024-09-03,1.8759e6,1.8787e6,1.8786e6,1.8745e6,-1324.519057,4018.695079,0.998566,0.002139


In [39]:
average_turnover = calculate_average_turnover(
    portfolio_turnover_table=portfolio_turnover_table
)
average_turnover

0.6663643113491394

In [40]:
annualized_turnover = calculate_annualized_turnover(
    portfolio_turnover_table=portfolio_turnover_table,
    rebalance_frequency=REBALANCE_FREQUENCY,
    rebalance_freq_unit=REBALANCE_FREQ_UNIT,
)
annualized_turnover

3.9981858680948363

In [41]:
total_transaction_cost = calculate_total_transaction_cost(
    portfolio_turnover_table=portfolio_turnover_table
)
total_transaction_cost

36064.927892391344

In [42]:
total_transaction_cost_to_initial_cap = calculate_total_transaction_cost_ratio(
    portfolio_turnover_table=portfolio_turnover_table, initial_capital=INITIAL_CAPITAL
)
total_transaction_cost_to_initial_cap

0.036064927892391345

In [43]:
average_transaction_cost_ratio = calculate_average_transaction_cost_ratio(
    portfolio_turnover_table=portfolio_turnover_table
)
average_transaction_cost_ratio

0.0014074921171406574

## Benchmark metrics

In [44]:
daily_benchmark_table = get_daily_benchmark_table(
    benchmark_data=benchmark_data,
    benchmark_ticker=BENCHMARK_TICKER,
    backtest_start_date=BACKTEST_START_DATE,
    backtest_end_date=end_date,
)
daily_benchmark_table

date,ticker,open,close,daily_return
date,str,f64,f64,f64
2021-12-31,"""^GSPC""",4775.209961,4766.180176,-0.001891
2022-01-03,"""^GSPC""",4778.140137,4796.560059,0.006374
2022-01-04,"""^GSPC""",4804.509766,4793.540039,-0.00063
2022-01-05,"""^GSPC""",4787.990234,4700.580078,-0.019393
2022-01-06,"""^GSPC""",4693.390137,4696.049805,-0.000964
…,…,…,…,…
2024-12-24,"""^GSPC""",5984.629883,6040.040039,0.011043
2024-12-26,"""^GSPC""",6024.970215,6037.589844,-0.000406
2024-12-27,"""^GSPC""",6006.169922,5970.839844,-0.011056


In [45]:
portfolio_benchmark_returns_table = get_portfolio_benchmark_returns_table(
    daily_benchmark_table=daily_benchmark_table,
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
portfolio_benchmark_returns_table

date,portfolio_return,benchmark_return
date,f64,f64
2021-12-31,0.000913,-0.001891
2022-01-03,-0.015445,0.006374
2022-01-04,-0.004386,-0.00063
2022-01-05,-0.04171,-0.019393
2022-01-06,-0.002663,-0.000964
…,…,…
2024-12-24,0.006932,0.011043
2024-12-26,-0.002668,-0.000406
2024-12-27,-0.007385,-0.011056


In [46]:
portfolio_beta = calculate_portfolio_beta(
    portfolio_benchmark_returns_table=portfolio_benchmark_returns_table
)
portfolio_beta

0.8889630611791157

In [47]:
jensens_alpha = calculate_jensens_alpha(
    portfolio_benchmark_returns_table=portfolio_benchmark_returns_table,
    portfolio_beta=portfolio_beta,
    annual_rf=ANNUAL_RISK_FREE_RATE,
)
jensens_alpha

0.16732425591559735

In [48]:
r_squared = calculate_r_squared(
    portfolio_benchmark_returns_table=portfolio_benchmark_returns_table
)
r_squared

0.39208567529044736

In [49]:
annualized_tracking_error = calculate_annualized_tracking_error(
    portfolio_benchmark_returns_table=portfolio_benchmark_returns_table
)
annualized_tracking_error

0.19445560014489463

In [50]:
information_ratio = calculate_information_ratio(
    portfolio_benchmark_returns_table=portfolio_benchmark_returns_table
)
information_ratio

0.8288571308477776

In [51]:
treynor_ratio = calculate_treynor_ratio(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df,
    portfolio_beta=portfolio_beta,
    annual_rf=ANNUAL_RISK_FREE_RATE,
)
treynor_ratio

0.243596151346317

In [52]:
best_day = get_best_day(daily_portfolio_value_return_df=daily_portfolio_value_return_df)
best_day

date,portfolio_value,daily_return
date,f64,f64
2024-07-31,1.7826e6,0.055222


In [53]:
worst_day = get_worst_day(daily_portfolio_value_return_df=daily_portfolio_value_return_df)
worst_day

date,portfolio_value,daily_return
date,f64,f64
2024-07-24,1.7295e6,-0.064894


In [54]:
win_rate = get_win_rate(
    daily_portfolio_value_return_df=daily_portfolio_value_return_df
)
win_rate

0.53315649867374